In [3]:
import pandas as pd


In [ ]:



# Load parquet files
benign_df = pd.read_parquet("benigndataset.parquet")
mal_df = pd.read_parquet("maliciousdataset.parquet")

print("Benign shape:", benign_df.shape)
print("Malware shape:", mal_df.shape)

# IMPORTANT: reset column names to be identical
benign_df.columns = range(benign_df.shape[1])
mal_df.columns = range(mal_df.shape[1])


# Concatenate safely
df = pd.concat([benign_df, mal_df], axis=0, ignore_index=True)

print("Final dataset shape:", df.shape)
print(df["label"].value_counts())

print(df.shape)


In [ ]:
X = df.drop("label", axis=1)
y = df["label"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_binary = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model_binary.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model_binary.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))



In [ ]:
import shap

explainer = shap.TreeExplainer(model_binary)
shap_values = explainer.shap_values(X_test.iloc[:100])

# Global explanation
shap.summary_plot(shap_values[1], X_test.iloc[:100])


In [ ]:
sample = X_test.iloc[0:1]
pred = model_binary.predict(sample)

if pred[0] == 1:
    print("Malware detected")
else:
    print("Benign app")
